In [ ]:
import os
import json
from yt_dlp import YoutubeDL
from pydub import AudioSegment, silence
import whisper
from spleeter.separator import Separator
from spleeter.audio.adapter import AudioAdapter

os.makedirs('data/youtube-audio', exist_ok=True)
os.makedirs('data/youtube-vocal', exist_ok=True)
os.makedirs('data/youtube-text', exist_ok=True)


In [ ]:
video_id = "BTe8pFl3oDo"


In [ ]:
ydl_opts = {
    'outtmpl': 'data/youtube-audio/%(id)s.%(ext)s',
    'format': 'mp3/bestaudio/best',
    'postprocessors': [{
        'key': 'FFmpegExtractAudio',
        'preferredcodec': 'wav',
    }]
}

with YoutubeDL(ydl_opts) as ydl:
    ydl.download(["https://www.youtube.com/watch?v=%s" % video_id])


In [ ]:
audio_path = "data/youtube-audio/%s.wav" % video_id
vocal_path = "data/youtube-vocal/"

separator = Separator('spleeter:2stems')
separator.separate_to_file(audio_path, vocal_path)


In [ ]:
# 無音部分でファイルを分割する
# https://github.com/jiaaro/pydub/
# https://github.com/jiaaro/pydub/blob/master/API.markdown#silencesplit_on_silence


file = AudioSegment.from_wav("data/youtube-vocal/%s/vocals.wav" % video_id)

chunks = silence.split_on_silence(
    file,
    min_silence_len=3000,
    silence_thresh=-40,
    seek_step=1000,
)

for i, chunk in enumerate(chunks):
    chunk_path = "data/youtube-audio-chunk/%s_%04d.wav" % (video_id, i + 1)
    chunk.export(chunk_path, format="wav")


In [ ]:
whisper_model = whisper.load_model("medium")


In [ ]:
audio_directory = "data/youtube-audio-chunk"
text_directory = "data/youtube-text-chunk"

for file in sorted(os.listdir(audio_directory)):
    audio_path = os.path.join(audio_directory, file)
    text_path = os.path.join(text_directory, file.replace(".wav", ".json"))

    result = whisper_model.transcribe(audio_path, verbose=True, language="ja")

    with open(text_path, 'w') as outfile:
        json.dump(result, outfile, indent=4, ensure_ascii=False)
